In [3]:
%pip install azure-ai-ml azure-identity scikit-learn joblib pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install "numpy<2" "scikit-learn>=1.3,<1.5"
%pip install --no-cache-dir --force-reinstall scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 3.3 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 2.5 MB/s  0:00:04m0:00:010:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: scikit-learn━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: scikit-learn 1.2.232m0/2 [numpy]
    Uninstalling scikit-learn-1.2.2:━━━━━━━━ 0/2 [numpy]
      Successfully uninstalled scikit-learn-1.2.2━━━━━━━━━━━━━━━━━ 1/2 [scikit-learn]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 4.3 MB/s  0:00:02 eta 0:00:01m
   ━━━━━━━━━━━━━━━

In [1]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()

X = iris.data
y = iris.target

print("Features shape:", X.shape)
print("Labels shape:", y.shape)

Features shape: (150, 4)
Labels shape: (150,)


In [2]:
from sklearn.linear_model import LogisticRegression
import joblib

model = LogisticRegression(max_iter=200)
model.fit(X, y)

joblib.dump(model, "iris_model.joblib")
print("Iris model trained")

Iris model trained


In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Replace with your details
SUBSCRIPTION_ID = "b3151994-1713-46ca-be03-528439a42153"
RESOURCE_GROUP = "iris-ml"
WORKSPACE_NAME = "iris-ml-ws"

credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)

print(f"Connected to workspace: {ml_client.workspace_name}")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Connected to workspace: iris-ml-ws


In [4]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model = ml_client.models.create_or_update(
    Model(
        name="iris-classifier",
        path="iris_model.joblib",
        type=AssetTypes.CUSTOM_MODEL,
        description="Iris classification model"
    )
)

print(f"Model registered: {model.name}")
print(f"Model version: {model.version}")

Uploading iris_model.joblib (< 1 MB): 100%|██████████| 991/991 [00:00<00:00, 17.6kB/s]




Model registered: iris-classifier
Model version: 2


In [5]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="sklearn-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file={
        "name": "sklearn-env",
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.9",
            "scikit-learn=1.2.2",
            "joblib",
            "numpy",
            "pip"
        ]
    }
)

env = ml_client.environments.create_or_update(env)
print(f"Environment created: {env.name}")

Environment created: sklearn-env


In [6]:
# Get workspace details correctly
workspace = ml_client.workspaces.get(name=WORKSPACE_NAME)
print("Workspace location:", workspace.location)

Workspace location: southindia


In [13]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import time
import random

# Generate unique name with timestamp and random number
unique_id = f"{int(time.time())}{random.randint(100,999)}"
endpoint_name = f"irisep{unique_id}"  # Keep it short, no hyphens

print(f"Creating endpoint: {endpoint_name}")

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Iris classification endpoint",
    auth_mode="key"
)

# Create with longer timeout
operation = ml_client.online_endpoints.begin_create_or_update(endpoint)
print("Endpoint creation started... waiting...")
endpoint = operation.result(timeout=600)  # 10 minute timeout

print(f"Endpoint created: {endpoint.name}")
print(f"Endpoint state: {endpoint.provisioning_state}")

Creating endpoint: irisep1770272926229
Endpoint creation started... waiting...
Endpoint created: irisep1770272926229
Endpoint state: Succeeded


In [ ]:
# Try these in order:
instance_types = [
    "Standard_F2s_v2",
    "Standard_E2s_v3", 
    "Standard_DS2_v2",
    "Standard_D2s_v3",
    "Standard_B2s"
]

In [20]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="iris-deployment",
    endpoint_name=endpoint_name,
    model=model,
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_B2s",  # Changed instance type
    instance_count=1
)

deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment created: {deployment.name}")

Check: endpoint irisep1770272926229 exists


Uploading Azure (0.03 MBs): 100%|██████████| 33518/33518 [00:00<00:00, 226071.27it/s]




HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"VmSize\":[\"The specified SKU 'Standard_B2s' is not supported. More details at https://docs.microsoft.com/en-us/azure/machine-learning/reference-managed-online-endpoints-vm-sku-list\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-7263db323e74ddb86fa7b5a84f46fb80-91c4872cfcf77111-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"VmSize\":[\"The specified SKU 'Standard_B2s' is not supported. More details at https://docs.microsoft.com/en-us/azure/machine-learning/reference-managed-online-endpoints-vm-sku-list\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors o